In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'

In [ ]:
var_dims_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/txt/dim_red/var_dims.txt'
var_dims = pd.read_csv(var_dims_path, header=None)[0].values
var_dims

In [ ]:
v = var_dims[0]
tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/{v}/merged.processed.small.h5ad')
final_cells = tmp_adata.obs.index
final_cells.shape

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', header=0, index_col=0)
clusters.shape

### load all of the feature sets

In [ ]:
adatas = []
for v in var_dims:
    tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/{v}/merged.processed.small.h5ad')
    tmp_adata.obs['cluster'] = clusters.reindex(tmp_adata.obs.index)['pred_coarse']
    tmp_adata = tmp_adata[tmp_adata.obs.index.isin(final_cells)]
    adatas.append(tmp_adata)

In [ ]:
train_size = 0.8
test_size = 0.2
model = OneVsRestClassifier(XGBClassifier(eval_metric='logloss', random_state=42))

In [ ]:
tmp_adata = adatas[0]
X = np.hstack([tmp_adata.obsm['X_pca'], tmp_adata.obs['time_num'].values.reshape(-1, 1)])
y = tmp_adata.obs['donor']

X_train, X_test, y_train, y_test_tmp = train_test_split(X, y, test_size=test_size, train_size=train_size)
model.fit(X_train, y_train)

tmp_pred = model.predict(X_test)
print(classification_report(y_test_tmp, tmp_pred))

In [ ]:
f1_list = []
for tmp_adata in adatas:
    X = np.hstack([tmp_adata.obsm['X_pca'], tmp_adata.obs['time_num'].values.reshape(-1, 1)])
    y = tmp_adata.obs['donor']

    X_train, X_test, y_train, y_test_tmp = train_test_split(X, y, test_size=test_size, train_size=train_size)
    model.fit(X_train, y_train)

    tmp_pred = model.predict(X_test)
    print(classification_report(y_test_tmp, tmp_pred))
    
    tmp_f1 = f1_score(y_test_tmp, tmp_pred, average=None)
    f1_list.append(tmp_f1)

In [ ]:
donors = tmp_adata.obs['donor'].unique()

In [ ]:
final_scores = pd.DataFrame(f1_list)
final_scores.columns = donors
final_scores.index = var_dims

In [ ]:
final_scores.head()

In [ ]:
final_scores['var_dim'] = final_scores.index

In [ ]:
to_plot = final_scores.melt(id_vars='var_dim')
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 5))

x_lab = 'variable'
y_lab = 'value'
hue = 'var_dim'

ax = sns.barplot(data=to_plot, x=x_lab, y=y_lab, hue=hue)

ax.set_xlabel('F1-Score', fontsize=16)
ax.set_ylabel('Donor', fontsize=16)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelsize=14, labelrotation=45)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
to_plot.to_csv(f'{outdir}/figure2/snmct_mcg_donor_feature_f1.csv', sep='\t')

In [ ]:
!date